In [ ]:
import pandas as pd
import glob

files = glob.glob("../data/raw/*.json")
streams = pd.concat([pd.read_json(file) for file in files])
streams.head()

In [ ]:
streams.rename(columns={"master_metadata_track_name": "track_name"}, inplace=True)
streams.rename(columns={"master_metadata_album_artist_name": "artist_name"}, inplace=True)
streams.rename(columns={"master_metadata_album_album_name": "album_name"}, inplace=True)

streams = streams.loc[:, ["ts", "ms_played", "spotify_track_uri","track_name", "artist_name", "album_name"]]

streams["ts"] = streams["ts"].map(lambda p: pd.to_datetime(p)) #just editing ts column 

In [ ]:
artist_streams = streams.groupby("artist_name").ms_played.sum().sort_values(ascending=False)
artist_streams.head(10)

In [ ]:

streams["differences"] = streams["ts"].diff().dt.total_seconds() - streams["ms_played"] / 1000
streams["differences"] = streams["differences"].map(lambda x: 0 if x < 0 else x)
# print(differences.head())
streams["differences"][:1000].plot(logy = True)


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

streams["differences"] = streams["differences"].fillna(1000)
space = np.logspace(0, 6, 50)
streams["differences"].plot.hist(bins = space)
plt.xscale("log")
plt.yscale("log")

In [ ]:
streams["session_num"] = (streams["differences"] > 600).cumsum()
print(streams.head())

In [ ]:
streams = streams.loc[streams["ms_played"] > 30000] 
sessions = streams.groupby("session_num")
stats = sessions.size().describe()
print(stats)

In [ ]:
sessions = sessions["session_num"].transform("size")
print(sessions.head())

In [ ]:
# sesh_list = sessions["spotify_track_uri"].agg(list)
# print(sesh_list)